In [ ]:
/content/drive/MyDrive/PhD/Sentiment_model/Dataset/Book(Open)/Books.jsonl

### 读取 JSONL 文件

我们将使用指定的路径读取文件。这里展示两种方法：
1. 使用 `json` 模块逐行读取（适合检查数据结构）。
2. 使用 `pandas` 读取为 DataFrame（适合数据分析）。

In [ ]:
import json
import random
import csv
import os
from sklearn.model_selection import train_test_split

# ================= 配置 =================
BASE_PATH = '/content/drive/MyDrive/PhD/Sentiment_model/Dataset/Book(Open)/'
INPUT_FILE = os.path.join(BASE_PATH, 'Books.jsonl')
TRAIN_FILE = os.path.join(BASE_PATH, 'train.csv')
TEST_FILE  = os.path.join(BASE_PATH, 'test.csv')

TOTAL_SAMPLES = 70000
TARGET_RATINGS = {1.0, 2.0, 4.0, 5.0}
SAMPLES_PER_RATING = TOTAL_SAMPLES // len(TARGET_RATINGS)
# =======================================

# 🚫 定义 Pandas 可能会误读为 NaN 的“坏字符串”
BAD_STRINGS = {'nan', 'null', 'none', '', 'na', 'n/a'}

def is_valid_text(text):
    """
    超严格检查：如果是 None，或者 strip 后为空，
    或者内容是 'nan', 'null' 等会被 Pandas 误读的词，统统返回 False
    """
    if text is None:
        return False

    # 转字符串并去空格
    s = str(text).strip()

    # 1. 检查是不是空字符串
    if not s:
        return False

    # 2. 检查是不是 "nan", "null" (转小写对比)
    if s.lower() in BAD_STRINGS:
        return False

    # 3. (可选) 只有 1 个字符的文本通常也是垃圾数据，建议过滤
    if len(s) < 2:
        return False

    return True

def process_and_balance_data():
    print(f"🚀 开始处理: {INPUT_FILE}")
    print("🧹 启用【超严格】过滤模式...")

    reservoirs = {r: [] for r in TARGET_RATINGS}
    counters = {r: 0 for r in TARGET_RATINGS}
    dropped_count = 0

    try:
        with open(INPUT_FILE, 'r', encoding='utf-8') as f:
            for line_num, line in enumerate(f):
                line = line.strip()
                if not line: continue

                try:
                    data = json.loads(line)

                    if 'rating' not in data: continue
                    try:
                        rating = float(data['rating'])
                    except ValueError: continue

                    if rating in TARGET_RATINGS:
                        # 获取文本
                        text_content = data.get('text')

                        # === ⚡️ 核心修改：使用严格检查函数 ===
                        if not is_valid_text(text_content):
                            dropped_count += 1
                            continue # 👈 只要进到这里，下面 append 绝对不会执行
                        # ===================================

                        # 确保转换为干净的字符串
                        clean_text = str(text_content).strip()

                        # 蓄水池采样
                        item = {'text': clean_text, 'rating': rating}

                        if len(reservoirs[rating]) < SAMPLES_PER_RATING:
                            reservoirs[rating].append(item)
                        else:
                            idx = random.randint(0, counters[rating])
                            if idx < SAMPLES_PER_RATING:
                                reservoirs[rating][idx] = item

                        counters[rating] += 1

                except json.JSONDecodeError:
                    continue

                if (line_num + 1) % 200000 == 0:
                    print(f"   ⏳ 已扫描 {line_num + 1} 行 (已剔除 '坏' 数据: {dropped_count} 条)...")

    except FileNotFoundError:
        print(f"❌ 错误: 找不到文件 {INPUT_FILE}")
        return

    # --- 整合数据 ---
    all_data = []
    print(f"\n✅ 采样完成。共严格剔除了 {dropped_count} 条无效数据（含空值、NaN、Null等）。")

    for r in TARGET_RATINGS:
        # Label: 1,2 -> 0; 4,5 -> 1
        label = 0 if r <= 2 else 1
        for item in reservoirs[r]:
            all_data.append({
                'label': label,
                'text': item['text'],
                'rating': item['rating']
            })

    if not all_data: return

    # --- 切分数据 ---
    print(f"✂️ 正在划分 {len(all_data)} 条数据...")
    labels = [item['label'] for item in all_data]

    # 确保有数据可分
    if len(all_data) < 5:
        print("❌ 数据不足。"); return

    train_data, test_data = train_test_split(
        all_data, test_size=0.2, random_state=42, stratify=labels
    )

    # --- 写入 CSV ---
    def save_csv(path, dataset):
        with open(path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=['label', 'text', 'rating'])
            writer.writeheader()
            writer.writerows(dataset)
        print(f"💾 已保存: {path}")

    save_csv(TRAIN_FILE, train_data)
    save_csv(TEST_FILE, test_data)

    # --- 最终验证 ---
    print("\n🔎 最终检查 (如果不报错，说明完全无空值):")
    import pandas as pd
    try:
        df_check = pd.read_csv(TRAIN_FILE)
        # 检查是否还有空值
        null_sum = df_check['text'].isnull().sum()
        print(f"   Pandas读取训练集检查: 'text' 列空值数量 = {null_sum}")
        if null_sum > 0:
            print("   ⚠️ 依然发现空值，可能是特殊编码字符导致。")
        else:
            print("   ✅ 完美！Pandas 未检测到任何空值。")
    except Exception as e:
        print(f"   (无法自动验证，请手动检查: {e})")

process_and_balance_data()

🚀 开始处理: /content/drive/MyDrive/PhD/Sentiment_model/Dataset/Book(Open)/Books.jsonl
🧹 启用【超严格】过滤模式...
   ⏳ 已扫描 200000 行 (已剔除 '坏' 数据: 132 条)...
   ⏳ 已扫描 400000 行 (已剔除 '坏' 数据: 229 条)...
   ⏳ 已扫描 600000 行 (已剔除 '坏' 数据: 377 条)...
   ⏳ 已扫描 800000 行 (已剔除 '坏' 数据: 607 条)...
   ⏳ 已扫描 1000000 行 (已剔除 '坏' 数据: 818 条)...
   ⏳ 已扫描 1200000 行 (已剔除 '坏' 数据: 937 条)...
   ⏳ 已扫描 1400000 行 (已剔除 '坏' 数据: 1234 条)...
   ⏳ 已扫描 1600000 行 (已剔除 '坏' 数据: 1456 条)...
   ⏳ 已扫描 1800000 行 (已剔除 '坏' 数据: 1612 条)...
   ⏳ 已扫描 2000000 行 (已剔除 '坏' 数据: 1759 条)...
   ⏳ 已扫描 2200000 行 (已剔除 '坏' 数据: 2143 条)...
   ⏳ 已扫描 2400000 行 (已剔除 '坏' 数据: 2288 条)...
   ⏳ 已扫描 2600000 行 (已剔除 '坏' 数据: 2535 条)...
   ⏳ 已扫描 2800000 行 (已剔除 '坏' 数据: 2693 条)...
   ⏳ 已扫描 3000000 行 (已剔除 '坏' 数据: 2912 条)...
   ⏳ 已扫描 3200000 行 (已剔除 '坏' 数据: 3051 条)...
   ⏳ 已扫描 3400000 行 (已剔除 '坏' 数据: 3213 条)...
   ⏳ 已扫描 3600000 行 (已剔除 '坏' 数据: 3347 条)...
   ⏳ 已扫描 3800000 行 (已剔除 '坏' 数据: 3512 条)...
   ⏳ 已扫描 4000000 行 (已剔除 '坏' 数据: 3847 条)...
   ⏳ 已扫描 4200000 行 (已剔除 '坏' 数据: 3933 条)...
   ⏳ 已扫描

In [ ]:
import json
import random
import csv
import os
from sklearn.model_selection import train_test_split

# ================= 1. 配置区域 =================
# 基础路径
BASE_PATH = '/content/drive/MyDrive/PhD/Sentiment_model/Dataset/Book(Open)/'
INPUT_FILE = os.path.join(BASE_PATH, 'Books.jsonl')
TRAIN_FILE = os.path.join(BASE_PATH, 'train.csv')
TEST_FILE  = os.path.join(BASE_PATH, 'test.csv')

# 采样配置
TOTAL_SAMPLES = 70000
TARGET_RATINGS = {1.0, 2.0, 4.0, 5.0}
SAMPLES_PER_RATING = TOTAL_SAMPLES // len(TARGET_RATINGS) # 17500 per rating
# ==============================================

def process_and_balance_data():
    print(f"🚀 开始读取文件: {INPUT_FILE}")
    print(f"🎯 目标字段: 'rating' (值: {TARGET_RATINGS})")

    # 初始化蓄水池和计数器
    reservoirs = {r: [] for r in TARGET_RATINGS}
    counters = {r: 0 for r in TARGET_RATINGS}

    try:
        with open(INPUT_FILE, 'r', encoding='utf-8') as f:
            for line_num, line in enumerate(f):
                line = line.strip()
                if not line: continue

                try:
                    data = json.loads(line)

                    # 1. 精确获取 rating
                    if 'rating' not in data:
                        # 尝试兼容字符串类型的数字，例如 "5"
                        continue

                    try:
                        rating = float(data['rating'])
                    except ValueError:
                        continue # 如果 rating 不是数字，跳过

                    # 2. 筛选目标分数 (1, 2, 4, 5)
                    if rating in TARGET_RATINGS:

                        # 3. 获取文本 (优先取 'text'，如果没有则取 'reviewText' + 'summary')
                        # 根据您的描述，文件中可能有直接的 'text' 列
                        if 'text' in data:
                            text_content = data['text']
                        else:
                            # 备用方案：如果没有 'text' 字段，尝试拼接常见字段
                            review = data.get('reviewText', '')
                            summary = data.get('summary', '')
                            text_content = f"{summary} {review}".strip()

                        if not text_content: continue

                        # 4. 蓄水池采样逻辑 (保证大文件内存不溢出)
                        item = {'text': text_content, 'rating': rating}

                        if len(reservoirs[rating]) < SAMPLES_PER_RATING:
                            reservoirs[rating].append(item)
                        else:
                            idx = random.randint(0, counters[rating])
                            if idx < SAMPLES_PER_RATING:
                                reservoirs[rating][idx] = item

                        counters[rating] += 1

                except json.JSONDecodeError:
                    continue

                if (line_num + 1) % 200000 == 0:
                    print(f"   ⏳ 已扫描 {line_num + 1} 行...")

    except FileNotFoundError:
        print(f"❌ 错误: 找不到文件 {INPUT_FILE}")
        return

    # --- 第二步：整合数据 ---
    all_data = []
    print("\n✅ 采样完成。正在生成标签...")

    total_collected = 0
    for r in TARGET_RATINGS:
        count = len(reservoirs[r])
        total_collected += count
        print(f"   Rating {r}: {count} 条")

        # 生成 Label: 1,2 -> 0; 4,5 -> 1
        label = 0 if r <= 2 else 1

        for item in reservoirs[r]:
            # 只保留需要的字段
            all_data.append({
                'label': label,
                'text': item['text'],
                'rating': item['rating']
            })

    if total_collected == 0:
        print("\n❌ [错误] 未提取到任何数据。")
        print("   请再次确认 JSON 文件中是否确实包含 key 为 'rating' 的字段。")
        return

    # --- 第三步：分层切分 (Stratified Split) ---
    print(f"\n✂️ 正在以 8:2 比例划分 {len(all_data)} 条数据...")

    labels = [item['label'] for item in all_data]

    train_data, test_data = train_test_split(
        all_data,
        test_size=0.2,
        random_state=42,
        stratify=labels  # 关键：保证训练集和测试集的 label 比例严格一致
    )

    # --- 第四步：写入 CSV (只写 label, text, rating) ---
    def save_csv(path, dataset):
        # 确保输出目录存在
        os.makedirs(os.path.dirname(path), exist_ok=True)

        with open(path, 'w', newline='', encoding='utf-8') as f:
            # 指定列名顺序
            fieldnames = ['label', 'text', 'rating']
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(dataset)
        print(f"💾 已保存: {path} (包含 {len(dataset)} 行)")

    save_csv(TRAIN_FILE, train_data)
    save_csv(TEST_FILE, test_data)

    # 验证最终数量
    print("\n📈 最终分布验证 (确保 label 0 和 1 数量相等):")
    tr_0 = sum(1 for x in train_data if x['label'] == 0)
    tr_1 = sum(1 for x in train_data if x['label'] == 1)
    te_0 = sum(1 for x in test_data if x['label'] == 0)
    te_1 = sum(1 for x in test_data if x['label'] == 1)

    print(f"   训练集: Label 0 = {tr_0}, Label 1 = {tr_1}")
    print(f"   测试集: Label 0 = {te_0}, Label 1 = {te_1}")

# 运行
process_and_balance_data()

🚀 开始读取文件: /content/drive/MyDrive/PhD/Sentiment_model/Dataset/Book(Open)/Books.jsonl
🎯 目标字段: 'rating' (值: {1.0, 2.0, 4.0, 5.0})
   ⏳ 已扫描 200000 行...
   ⏳ 已扫描 400000 行...
   ⏳ 已扫描 600000 行...
   ⏳ 已扫描 800000 行...
   ⏳ 已扫描 1000000 行...
   ⏳ 已扫描 1200000 行...
   ⏳ 已扫描 1400000 行...
   ⏳ 已扫描 1600000 行...
   ⏳ 已扫描 1800000 行...
   ⏳ 已扫描 2000000 行...
   ⏳ 已扫描 2200000 行...
   ⏳ 已扫描 2400000 行...
   ⏳ 已扫描 2600000 行...
   ⏳ 已扫描 2800000 行...
   ⏳ 已扫描 3000000 行...
   ⏳ 已扫描 3200000 行...
   ⏳ 已扫描 3400000 行...
   ⏳ 已扫描 3600000 行...
   ⏳ 已扫描 3800000 行...
   ⏳ 已扫描 4000000 行...
   ⏳ 已扫描 4200000 行...
   ⏳ 已扫描 4400000 行...
   ⏳ 已扫描 4600000 行...
   ⏳ 已扫描 4800000 行...
   ⏳ 已扫描 5000000 行...
   ⏳ 已扫描 5200000 行...
   ⏳ 已扫描 5400000 行...
   ⏳ 已扫描 5600000 行...
   ⏳ 已扫描 5800000 行...
   ⏳ 已扫描 6000000 行...
   ⏳ 已扫描 6200000 行...
   ⏳ 已扫描 6400000 行...
   ⏳ 已扫描 6600000 行...
   ⏳ 已扫描 6800000 行...
   ⏳ 已扫描 7000000 行...
   ⏳ 已扫描 7200000 行...
   ⏳ 已扫描 7400000 行...
   ⏳ 已扫描 7600000 行...
   ⏳ 已扫描 7800000 行...
   ⏳ 已扫描 8000000 行.